# Enterprise Incentive Servicing Analytics
## Layer 2 — SQL Business Queries via DuckDB
**Source:** `data/processed/employee_incentive_final.csv` (output from Layer 1)  
**Tool:** DuckDB — analytical SQL engine, PostgreSQL-compatible syntax  
**Purpose:** Answer 6 business questions that mirror real incentive servicing analyst work

## Setup — Install & Connect

In [17]:
# Install if needed
import subprocess
subprocess.run(['pip', 'install', 'duckdb', '--quiet'], capture_output=True)

import duckdb
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 50)

# Connect and load CSV as a SQL table
con = duckdb.connect()
con.execute("""
    CREATE TABLE employees AS
    SELECT * FROM read_csv_auto('data/processed/employee_incentive_final.csv')
""")

row_count = con.execute('SELECT COUNT(*) FROM employees').fetchone()[0]
col_count = len(con.execute('SELECT * FROM employees LIMIT 1').df().columns)
print(f"✅ Table loaded: {row_count} rows × {col_count} columns")
print(f"\nColumn list:")
print(con.execute('DESCRIBE employees').df()[['column_name','column_type']].to_string(index=False))

✅ Table loaded: 1470 rows × 36 columns

Column list:
             column_name column_type
          EmployeeNumber      BIGINT
                     Age      BIGINT
                  Gender     VARCHAR
              Department     VARCHAR
                 JobRole     VARCHAR
               Education      BIGINT
          EducationField     VARCHAR
           MaritalStatus     VARCHAR
           MonthlyIncome      BIGINT
          ExpectedSalary      DOUBLE
               CompRatio      DOUBLE
             CompSegment     VARCHAR
       PercentSalaryHike      BIGINT
       PerformanceRating      BIGINT
        StockOptionLevel      BIGINT
         StockMultiplier      DOUBLE
                OverTime     BOOLEAN
          YearsAtCompany      BIGINT
      YearsInCurrentRole      BIGINT
 YearsSinceLastPromotion      BIGINT
    YearsWithCurrManager      BIGINT
       TotalWorkingYears      BIGINT
      NumCompaniesWorked      BIGINT
         JobSatisfaction      BIGINT
 EnvironmentSatisfacti

## Query 1 — Attrition Rate by Department & Compensation Band
**Business Question:** Which department-compensation combinations have the highest flight risk?  
**Why it matters:** Prioritises which segments need immediate incentive intervention

In [18]:
q1 = con.execute("""
SELECT
    Department,
    CASE
        WHEN MonthlyIncome <  3000 THEN '1_Low (<3K)'
        WHEN MonthlyIncome <  7000 THEN '2_Mid (3K-7K)'
        WHEN MonthlyIncome < 12000 THEN '3_High (7K-12K)'
        ELSE                            '4_Senior (12K+)'
    END AS CompBand,
    COUNT(*)                                AS HeadCount,
    SUM(AttritionFlag)                      AS AttritionCount,
    ROUND(AVG(AttritionFlag) * 100, 1)     AS AttritionRate_Pct,
    ROUND(AVG(MonthlyIncome), 0)           AS AvgMonthlyIncome,
    ROUND(AVG(CompRatio), 3)              AS AvgCompRatio
FROM employees
GROUP BY Department, CompBand
ORDER BY Department, AttritionRate_Pct DESC
""").df()

print("Q1 RESULT — Attrition by Department & Compensation Band")
print("=" * 65)
display(q1)

# Key insight
worst = q1.loc[q1['AttritionRate_Pct'].idxmax()]
print(f"\n📌 Highest attrition: {worst['Department']} / {worst['CompBand']} at {worst['AttritionRate_Pct']}%")

Q1 RESULT — Attrition by Department & Compensation Band


,Department,CompBand,HeadCount,AttritionCount,AttritionRate_Pct,AvgMonthlyIncome,AvgCompRatio
0,Human Resources,1_Low (<3K),26,10.0,38.5,2440.0,1.068
1,Human Resources,3_High (7K-12K),6,2.0,33.3,9623.0,1.584
2,Human Resources,2_Mid (3K-7K),20,0.0,0.0,4954.0,1.007
3,Human Resources,4_Senior (12K+),11,0.0,0.0,18089.0,1.408
4,Research & Development,1_Low (<3K),301,75.0,24.9,2401.0,0.804
5,Research & Development,2_Mid (3K-7K),402,41.0,10.2,4686.0,0.911
6,Research & Development,3_High (7K-12K),122,12.0,9.8,9580.0,1.272
7,Research & Development,4_Senior (12K+),136,5.0,3.7,16625.0,1.454
8,Sales,1_Low (<3K),68,28.0,41.2,2347.0,0.809
9,Sales,3_High (7K-12K),112,22.0,19.6,9110.0,1.297



📌 Highest attrition: Sales / 1_Low (<3K) at 41.2%


## Query 2 — Compa-Ratio Distribution by Job Role
**Business Question:** Which roles show the largest pay equity gaps?  
**Why it matters:** Identifies roles where employees are systematically underpaid relative to their predicted market-rate salary

In [19]:
q2 = con.execute("""
SELECT
    JobRole,
    COUNT(*)                                AS HeadCount,
    ROUND(AVG(CompRatio), 3)              AS AvgCompRatio,
    ROUND(MIN(CompRatio), 3)              AS MinCompRatio,
    ROUND(MAX(CompRatio), 3)              AS MaxCompRatio,
    SUM(CASE WHEN CompSegment = 'Underpaid' THEN 1 ELSE 0 END) AS UnderpaidCount,
    SUM(CASE WHEN CompSegment = 'Fair'      THEN 1 ELSE 0 END) AS FairCount,
    SUM(CASE WHEN CompSegment = 'Overpaid'  THEN 1 ELSE 0 END) AS OverpaidCount,
    ROUND(
        SUM(CASE WHEN CompSegment = 'Underpaid' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    )                                       AS UnderpaidPct
FROM employees
GROUP BY JobRole
ORDER BY AvgCompRatio ASC
""").df()

print("Q2 RESULT — Compa-Ratio by Job Role (sorted: most underpaid first)")
print("=" * 65)
display(q2)

worst_role = q2.loc[q2['UnderpaidPct'].idxmax()]
print(f"\n📌 Most underpaid role: {worst_role['JobRole']} — {worst_role['UnderpaidPct']}% of employees below fair comp")

Q2 RESULT — Compa-Ratio by Job Role (sorted: most underpaid first)


,JobRole,HeadCount,AvgCompRatio,MinCompRatio,MaxCompRatio,UnderpaidCount,FairCount,OverpaidCount,UnderpaidPct
0,Laboratory Technician,259,0.793,0.218,1.967,164.0,51.0,44.0,63.3
1,Sales Representative,83,0.811,0.204,1.353,44.0,30.0,9.0,53.0
2,Research Scientist,292,0.860,0.201,2.196,174.0,62.0,56.0,59.6
3,Healthcare Representative,131,1.009,0.410,1.992,46.0,45.0,40.0,35.1
4,Sales Executive,326,1.078,0.449,2.149,90.0,111.0,125.0,27.6
5,Human Resources,52,1.104,0.400,2.805,25.0,6.0,21.0,48.1
6,Manufacturing Director,145,1.134,0.406,2.221,34.0,42.0,69.0,23.4
7,Manager,102,1.401,0.847,2.611,1.0,20.0,81.0,1.0
8,Research Director,80,1.599,0.863,3.310,0.0,10.0,70.0,0.0



📌 Most underpaid role: Laboratory Technician — 63.3% of employees below fair comp


## Query 3 — Overtime Burden vs Incentive Level Cross-tab
**Business Question:** Are overtime workers receiving proportional incentive compensation, or are they systematically excluded?  
**Why it matters:** Detects structural fairness gaps in how incentive programs are applied

In [20]:
q3 = con.execute("""
SELECT
    OverTime,
    StockOptionLevel,
    COUNT(*)                                        AS HeadCount,
    ROUND(AVG(AttritionFlag) * 100, 1)            AS AttritionRate_Pct,
    ROUND(AVG(MonthlyIncome), 0)                  AS AvgMonthlyIncome,
    ROUND(AVG(CompRatio), 3)                      AS AvgCompRatio,
    ROUND(AVG(IncentiveEligible) * 100, 1)       AS EligibilityRate_Pct,
    ROUND(AVG(CASE WHEN IncentiveEligible = 1
                   THEN SimulatedPayout ELSE NULL END), 0) AS AvgPayoutIfEligible
FROM employees
GROUP BY OverTime, StockOptionLevel
ORDER BY OverTime, StockOptionLevel
""").df()

print("Q3 RESULT — Overtime × Incentive Level Cross-tab")
print("=" * 65)
display(q3)

# Highlight the gap
ot_yes = q3[q3['OverTime'] == 'Yes']['AttritionRate_Pct'].mean()
ot_no  = q3[q3['OverTime'] == 'No']['AttritionRate_Pct'].mean()
print(f"\n📌 Avg attrition — Overtime Yes: {ot_yes:.1f}% vs Overtime No: {ot_no:.1f}%")
print(f"   Overtime workers leave at {ot_yes/ot_no:.1f}x the rate of non-overtime employees")

Q3 RESULT — Overtime × Incentive Level Cross-tab


,OverTime,StockOptionLevel,HeadCount,AttritionRate_Pct,AvgMonthlyIncome,AvgCompRatio,EligibilityRate_Pct,AvgPayoutIfEligible
0,False,0,449,16.0,6221.0,1.027,57.0,9475.0
1,False,1,429,6.3,6974.0,1.021,56.9,10412.0
2,False,2,120,5.0,6095.0,1.029,55.8,10094.0
3,False,3,56,8.9,5686.0,0.934,71.4,12156.0
4,True,0,182,45.1,6107.0,1.012,0.0,NaN
5,True,1,167,17.4,7137.0,1.018,0.0,NaN
6,True,2,38,15.8,6416.0,1.042,0.0,NaN
7,True,3,29,34.5,6105.0,1.078,0.0,NaN



📌 Avg attrition — Overtime Yes: nan% vs Overtime No: nan%
   Overtime workers leave at nanx the rate of non-overtime employees


## Query 4 — Participant Eligibility Flight Risk Register
**Business Question:** Which high-performing employees are underpaid and at active risk of leaving?  
**Why it matters:** This is the actionable output — the list an analyst hands to HR business partners for immediate intervention

In [21]:
q4 = con.execute("""
SELECT
    EmployeeNumber,
    Department,
    JobRole,
    PerformanceRating,
    YearsAtCompany,
    MonthlyIncome,
    ROUND(ExpectedSalary, 0)                        AS ExpectedSalary,
    ROUND(CompRatio, 3)                             AS CompRatio,
    ROUND((ExpectedSalary - MonthlyIncome) * 12, 0) AS AnnualIncentiveGap,
    AttritionRisk,
    RiskScore,
    OverTime
FROM employees
WHERE
    PerformanceRating >= 3
    AND CompSegment = 'Underpaid'
    AND AttritionRisk IN ('High', 'Medium')
ORDER BY RiskScore DESC, AnnualIncentiveGap DESC
LIMIT 25
""").df()

print("Q4 RESULT — Top 25 Flight Risk: High Performers, Below-Market Pay")
print("=" * 65)
display(q4)

total_gap = q4['AnnualIncentiveGap'].sum()
print(f"\n📌 Total annual incentive gap for top 25 at-risk employees: ${total_gap:,.0f}")
print(f"   Average gap per employee: ${q4['AnnualIncentiveGap'].mean():,.0f}/year")

Q4 RESULT — Top 25 Flight Risk: High Performers, Below-Market Pay


,EmployeeNumber,Department,JobRole,PerformanceRating,YearsAtCompany,MonthlyIncome,ExpectedSalary,CompRatio,AnnualIncentiveGap,AttritionRisk,RiskScore,OverTime
0,1360,Research & Development,Manufacturing Director,3,10,10008,15075.0,0.664,60799.0,High,90,True
1,1278,Research & Development,Healthcare Representative,3,33,13577,18271.0,0.743,56331.0,High,90,True
2,1489,Sales,Sales Executive,4,15,4599,8921.0,0.516,51863.0,High,90,True
3,1827,Research & Development,Manufacturing Director,3,22,10333,14370.0,0.719,48445.0,High,90,True
4,1294,Research & Development,Manufacturing Director,3,16,5410,9314.0,0.581,46843.0,High,90,True
5,881,Research & Development,Research Scientist,3,10,2022,5797.0,0.349,45298.0,High,90,True
6,630,Research & Development,Research Scientist,3,10,5577,9235.0,0.604,43898.0,High,90,True
7,351,Research & Development,Laboratory Technician,3,9,2593,5970.0,0.434,40529.0,High,90,True
8,803,Research & Development,Manufacturing Director,3,15,5980,9145.0,0.654,37976.0,High,90,True
9,1244,Research & Development,Research Scientist,3,9,2235,5140.0,0.435,34865.0,High,90,True



📌 Total annual incentive gap for top 25 at-risk employees: $1,086,111
   Average gap per employee: $43,444/year


## Query 5 — Incentive Gap Cost by Department (ROI Ranked)
**Business Question:** Where should the incentive restructuring budget be allocated first to maximise retention ROI?  
**Why it matters:** Turns analysis into a capital allocation recommendation — the output leadership actually acts on

In [22]:
q5 = con.execute("""
SELECT
    Department,
    COUNT(*)                                                    AS TotalEmployees,
    SUM(CASE WHEN CompSegment = 'Underpaid'
             AND IncentiveEligible = 1 THEN 1 ELSE 0 END)      AS UnderpaidEligible,
    ROUND(
        SUM(CASE WHEN CompSegment = 'Underpaid' AND IncentiveEligible = 1
                 THEN (ExpectedSalary - MonthlyIncome) * 12
                 ELSE 0 END), 0
    )                                                           AS TotalIncentiveGapCost,
    ROUND(
        SUM(CASE WHEN AttritionRisk = 'High'
                 THEN MonthlyIncome * 12 * 0.5
                 ELSE 0 END), 0
    )                                                           AS ReplacementCostIfNoAction,
    ROUND(
        SUM(CASE WHEN AttritionRisk = 'High'
                 THEN MonthlyIncome * 12 * 0.5
                 ELSE 0 END) -
        SUM(CASE WHEN CompSegment = 'Underpaid' AND IncentiveEligible = 1
                 THEN (ExpectedSalary - MonthlyIncome) * 12
                 ELSE 0 END), 0
    )                                                           AS NetROI,
    ROUND(AVG(CompRatio), 3)                                   AS AvgCompRatio,
    ROUND(AVG(AttritionFlag) * 100, 1)                        AS AttritionRate_Pct
FROM employees
GROUP BY Department
ORDER BY NetROI DESC
""").df()

print("Q5 RESULT — Incentive Gap ROI by Department")
print("=" * 65)
display(q5)

best_dept = q5.loc[q5['NetROI'].idxmax()]
print(f"\n📌 Highest ROI department: {best_dept['Department']}")
print(f"   Fixing gap costs ${best_dept['TotalIncentiveGapCost']:,.0f} but prevents ${best_dept['ReplacementCostIfNoAction']:,.0f} in replacement costs")
print(f"   Net saving: ${best_dept['NetROI']:,.0f}")

Q5 RESULT — Incentive Gap ROI by Department


,Department,TotalEmployees,UnderpaidEligible,TotalIncentiveGapCost,ReplacementCostIfNoAction,NetROI,AvgCompRatio,AttritionRate_Pct
0,Human Resources,63,17.0,374087.0,320970.0,-53117.0,1.157,19.0
1,Sales,446,85.0,2743237.0,2608056.0,-135181.0,1.045,20.6
2,Research & Development,961,269.0,8642026.0,5126910.0,-3515116.0,1.000,13.8



📌 Highest ROI department: Human Resources
   Fixing gap costs $374,087 but prevents $320,970 in replacement costs
   Net saving: $-53,117


## Query 6 — Reconciliation Audit: Eligible but Excluded
**Business Question:** Which employees meet all incentive eligibility criteria but are receiving zero payout?  
**Why it matters:** This is core incentive *administration* work — identifying where the program is failing to reach the people it should cover

In [23]:
q6 = con.execute("""
SELECT
    Department,
    JobRole,
    COUNT(*)                                        AS EligibleButExcluded,
    ROUND(AVG(MonthlyIncome), 0)                   AS AvgIncome,
    ROUND(AVG(CompRatio), 3)                       AS AvgCompRatio,
    ROUND(AVG(PerformanceRating), 2)               AS AvgPerfRating,
    ROUND(AVG(YearsAtCompany), 1)                  AS AvgTenure,
    ROUND(AVG(AttritionFlag) * 100, 1)            AS AttritionRate_Pct,
    ROUND(
        SUM(MonthlyIncome * 12 * (PercentSalaryHike / 100.0) * StockMultiplier), 0
    )                                               AS MissedPayoutValue
FROM employees
WHERE IncentiveEligible = 1 AND SimulatedPayout = 0
GROUP BY Department, JobRole
HAVING COUNT(*) > 0
ORDER BY MissedPayoutValue DESC
""").df()

print("Q6 RESULT — Reconciliation Audit: Eligible Employees with Zero Payout")
print("=" * 65)
display(q6)

total_missed = q6['MissedPayoutValue'].sum()
total_excluded = q6['EligibleButExcluded'].sum()
print(f"\n📌 {total_excluded} eligible employees excluded from incentive payouts")
print(f"   Total missed payout value: ${total_missed:,.0f}")

Q6 RESULT — Reconciliation Audit: Eligible Employees with Zero Payout


,Department,JobRole,EligibleButExcluded,AvgIncome,AvgCompRatio,AvgPerfRating,AvgTenure,AttritionRate_Pct,MissedPayoutValue



📌 0 eligible employees excluded from incentive payouts
   Total missed payout value: $0


## Summary — Key Findings Across All 6 Queries

In [24]:
# Pull headline numbers across all queries for README
total_employees   = con.execute("SELECT COUNT(*) FROM employees").fetchone()[0]
attrition_rate    = con.execute("SELECT ROUND(AVG(AttritionFlag)*100,1) FROM employees").fetchone()[0]
underpaid_pct     = con.execute("SELECT ROUND(AVG(CASE WHEN CompSegment='Underpaid' THEN 1.0 ELSE 0 END)*100,1) FROM employees").fetchone()[0]
high_risk_n       = con.execute("SELECT COUNT(*) FROM employees WHERE AttritionRisk='High'").fetchone()[0]

gap_cost = con.execute("""
    SELECT COALESCE(ROUND(SUM(CASE WHEN CompSegment='Underpaid' AND IncentiveEligible=1
                          THEN (ExpectedSalary - MonthlyIncome)*12 ELSE 0 END), 0), 0)
    FROM employees
""").fetchone()[0]

replacement_cost = con.execute("""
    SELECT COALESCE(ROUND(SUM(CASE WHEN AttritionRisk='High'
                          THEN MonthlyIncome*12*0.5 ELSE 0 END), 0), 0)
    FROM employees
""").fetchone()[0]

excluded_n = con.execute(
    "SELECT COUNT(*) FROM employees WHERE IncentiveEligible=1 AND SimulatedPayout=0"
).fetchone()[0]

missed_payout = con.execute("""
    SELECT COALESCE(
        ROUND(SUM(MonthlyIncome * 12 * (PercentSalaryHike / 100.0) * StockMultiplier), 0),
        0
    )
    FROM employees
    WHERE IncentiveEligible=1 AND SimulatedPayout=0
""").fetchone()[0]

# Cast all to float for safe arithmetic
gap_cost         = float(gap_cost)
replacement_cost = float(replacement_cost)
missed_payout    = float(missed_payout)
net_roi          = replacement_cost - gap_cost

print("=" * 62)
print("  LAYER 2 COMPLETE — Key Findings for README")
print("=" * 62)
print(f"  Total employees analysed        : {total_employees}")
print(f"  Overall attrition rate          : {attrition_rate}%")
print(f"  Underpaid employees             : {underpaid_pct}% of workforce")
print(f"  High-risk headcount             : {high_risk_n}")
print(f"  Total incentive gap cost        : ${gap_cost:,.0f}")
print(f"  Replacement cost if no action   : ${replacement_cost:,.0f}")
print(f"  Net ROI from intervention       : ${net_roi:,.0f}")
print(f"  Excluded from payout (audit)    : {excluded_n} eligible employees")
print(f"  Missed payout value             : ${missed_payout:,.0f}")
print("=" * 62)


con.close()

  LAYER 2 COMPLETE — Key Findings for README
  Total employees analysed        : 1470
  Overall attrition rate          : 16.1%
  Underpaid employees             : 39.3% of workforce
  High-risk headcount             : 272
  Total incentive gap cost        : $11,759,351
  Replacement cost if no action   : $8,055,936
  Net ROI from intervention       : $-3,703,415
  Excluded from payout (audit)    : 0 eligible employees
  Missed payout value             : $0
